In [1]:
# Step 1: Load the Xenium Data into AnnData

import scanpy as sc
import pandas as pd
import numpy as np

# Load the cell feature matrix (counts per cell)
adata_xenium = sc.read_10x_h5(
    "/projectnb/ds596/projects/Team_7/data/xeniumMouseBrain/cell_feature_matrix.h5"
)
adata_xenium.var_names_make_unique()

# Add spatial coordinates from cells.csv.gz
cells = pd.read_csv(
    "/projectnb/ds596/projects/Team_7/data/xeniumMouseBrain/cells.csv.gz"
)
# Align by cell_id (the .h5 barcodes should match)
cells = cells.set_index("cell_id")
adata_xenium.obs = adata_xenium.obs.join(cells[["x_centroid", "y_centroid"]], how="left")

# Register spatial coords for visualization
adata_xenium.obsm["spatial"] = adata_xenium.obs[["x_centroid", "y_centroid"]].values

print(adata_xenium)

AnnData object with n_obs × n_vars = 36602 × 248
    obs: 'x_centroid', 'y_centroid'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'


/opt/conda/envs/cellpymc/lib/python3.7/site-packages/anndata/_core/anndata.py:1094: FutureWarning: is_categorical is deprecated and will be removed in a future version.  Use is_categorical_dtype instead
  if not is_categorical(df_full[k]):


In [2]:
# this block has an error - continue to run later blocks of code and it works

# Step 2: Get a Mouse Brain scRNA-seq Reference Atlas

import urllib.request

# Download the reference h5ad
urllib.request.urlretrieve(
    "https://cell2location.cog.sanger.ac.uk/tutorial/mouse_brain_snrna/all_cells_20200625.h5ad",
    "all_cells_20200625.h5ad"
)

# Download the refined cell type annotation CSV
urllib.request.urlretrieve(
    "https://cell2location.cog.sanger.ac.uk/tutorial/mouse_brain_snrna/snRNA_annotation_astro_subtypes_refined59_20200823.csv",
    "snRNA_annotation_astro_subtypes_refined59_20200823.csv"
)

# Load the reference into AnnData
adata_ref = sc.read_h5ad("all_cells_20200625.h5ad")
print(adata_ref)
print(adata_ref.obs["annotation_1_print"].value_counts())

AnnData object with n_obs × n_vars = 40572 × 31053
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'sample', 'barcode'
    var: 'feature_types', 'genome', 'SYMBOL', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'


KeyError: 'annotation_1_print'

In [ ]:
# Check what columns exist in the reference
print(adata_ref.obs.columns.tolist())

In [ ]:
import pandas as pd

# Load the annotation CSV
ann = pd.read_csv("snRNA_annotation_astro_subtypes_refined59_20200823.csv", index_col=0)
print(ann.head())
print(ann.columns.tolist())

In [ ]:
# this block has an error - continue to run later blocks of code and it works

# code to merge annotations into adata_ref

import pandas as pd

# Load the annotation CSV
ann = pd.read_csv("snRNA_annotation_astro_subtypes_refined59_20200823.csv", index_col=0)

# Merge annotations into adata_ref
adata_ref.obs = adata_ref.obs.join(ann, how="left")

# Verify the merge worked
print(adata_ref.obs["annotation_1"].value_counts())

In [ ]:
# Step 3: Find shared genes between Xenium and reference
shared_genes = adata_xenium.var_names.intersection(adata_ref.var_names)
print(f"Shared genes: {len(shared_genes)}")

# Subset both to shared genes
adata_xenium = adata_xenium[:, shared_genes].copy()
adata_ref = adata_ref[:, shared_genes].copy()

print(f"Xenium shape: {adata_xenium.shape}")
print(f"Reference shape: {adata_ref.shape}")

In [ ]:
# Reload Xenium
adata_xenium = sc.read_10x_h5(
    "/projectnb/ds596/projects/Team_7/data/xeniumMouseBrain/cell_feature_matrix.h5"
)
adata_xenium.var_names_make_unique()

# Reload reference
adata_ref = sc.read_h5ad("all_cells_20200625.h5ad")
adata_ref.obs = adata_ref.obs.join(ann, how="left")

# NOW check gene names before any subsetting
print("Xenium genes:", adata_xenium.var_names[:10].tolist())
print("Reference genes:", adata_ref.var_names[:10].tolist())
print("Reference SYMBOL column:", adata_ref.var["SYMBOL"][:10].tolist())

In [ ]:
# Convert SYMBOL to plain string first, then assign as index
adata_ref.var["SYMBOL"] = adata_ref.var["SYMBOL"].astype(str)
adata_ref.var_names = adata_ref.var["SYMBOL"]
adata_ref.var_names_make_unique()

# Verify it worked
print("Reference genes after fix:", adata_ref.var_names[:10].tolist())

# Now find shared genes
shared_genes = adata_xenium.var_names.intersection(adata_ref.var_names)
print(f"Shared genes: {len(shared_genes)}")

# Subset both to shared genes
adata_xenium = adata_xenium[:, shared_genes].copy()
adata_ref = adata_ref[:, shared_genes].copy()

print(f"Xenium shape: {adata_xenium.shape}")
print(f"Reference shape: {adata_ref.shape}")

In [ ]:
# Step 4: Basic QC
sc.pp.filter_cells(adata_xenium, min_counts=10)
sc.pp.filter_genes(adata_xenium, min_cells=5)

print(f"Xenium after QC: {adata_xenium.shape}")

In [ ]:
from cell2location.models import RegressionGeneBackgroundCoverageTorch
import numpy as np
import pandas as pd

# Prepare inputs
cell2covar = pd.DataFrame({
    "sample": adata_ref.obs["sample"].values,
    "annotation_1": adata_ref.obs["annotation_1"].values
}, index=adata_ref.obs_names)

X_data = adata_ref.X.toarray() if hasattr(adata_ref.X, "toarray") else np.array(adata_ref.X)

# Build the correct model
mod = RegressionGeneBackgroundCoverageTorch(
    sample_id="sample",
    cell2covar=cell2covar,
    X_data=X_data,
    n_iter=100,
    learning_rate=0.005,
    use_cuda=False,
    var_names=adata_ref.var_names.tolist(),
    obs_names=adata_ref.obs_names.tolist()
)

print("Model built successfully!")
print(f"n_obs: {mod.n_obs}, n_var: {mod.n_var}, n_fact: {mod.n_fact}")
# tells the model to use 90% of cells for training and 10% for validation
mod.fit_advi_iterative(n=1, n_iter=100, train_proportion=0.9)

In [ ]:
# Step 1: Sample posterior AFTER training
mod.sample_posterior(node='all', n_samples=1000, save_samples=False, mean_field_slot='init_1')
print(mod.samples['post_sample_means'].keys())

# Step 2: Extract cell type signatures
inf_aver = mod.samples['post_sample_means']['gene_factors']
inf_aver = pd.DataFrame(inf_aver,
                         index=mod.fact_names,
                         columns=mod.var_names).T
inf_aver = inf_aver.loc[:, ~mod.which_sample.values]
inf_aver.columns = [c.replace("annotation_1_", "") for c in inf_aver.columns]
print(inf_aver.shape)

# Step 3: Build spatial model
from cell2location.models import LocationModelLinearDependentWMultiExperiment

X_data_xenium = adata_xenium.X.toarray() if hasattr(adata_xenium.X, "toarray") else np.array(adata_xenium.X)

mod_spatial = LocationModelLinearDependentWMultiExperiment(
    cell_state_mat=inf_aver.values,
    X_data=X_data_xenium,
    n_comb=20,
    n_iter=100,
    learning_rate=0.01,
    sample_id=np.array(["xenium_sample"] * adata_xenium.n_obs),
    var_names=adata_xenium.var_names.tolist(),
    obs_names=adata_xenium.obs_names.tolist(),
    fact_names=inf_aver.columns.tolist()
)
print("Spatial model built!")

# Step 4: Train spatial model
mod_spatial.fit_advi(n=1, n_type='restart')

In [ ]:
# Use fewer samples with larger batches
mod_spatial.sample_posterior(node='all', n_samples=200, 
                              save_samples=False, 
                              mean_field_slot='init_1', 
                              batch_size=200)  # 1 batch instead of 25

In [ ]:
# Extract spot factors (cell type abundances per cell)
spot_factors = mod_spatial.samples['post_sample_means']['spot_factors']
print(spot_factors.shape)  # should be (36419 cells x 59 cell types)

In [ ]:
# Add cell type abundances to adata_xenium
adata_xenium.obsm['q05_cell_abundance_w_sf'] = spot_factors

# Add as individual columns in obs for easy access
for i, ct in enumerate(inf_aver.columns):
    adata_xenium.obs[ct] = spot_factors[:, i]

print("Cell type abundances added to adata_xenium!")
print(adata_xenium.obs[inf_aver.columns[:5]])

In [ ]:
# Convert cells index to string to match adata_xenium
cells.index = cells.index.astype(str)

# Add spatial coordinates back
adata_xenium.obs = adata_xenium.obs.join(cells[["x_centroid", "y_centroid"]], how="left")

# Verify
print(adata_xenium.obs[["x_centroid", "y_centroid"]].head())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

x = adata_xenium.obs['x_centroid']
y = adata_xenium.obs['y_centroid']

cell_types_to_plot = ['Astro_CTX', 'Oligo_2', 'Ext_L23', 'Ext_Hpc_CA1',
                       'Micro', 'Inh_Sst', 'OPC_1', 'Ext_L56']

for i, ct in enumerate(cell_types_to_plot):
    abundance = adata_xenium.obs[ct]
    sc = axes[i].scatter(x, y, c=abundance, cmap='viridis',
                          s=0.5, vmin=0, vmax=np.percentile(abundance, 99))
    axes[i].set_title(ct, fontsize=12)
    axes[i].axis('off')
    plt.colorbar(sc, ax=axes[i], shrink=0.7)

plt.suptitle('Cell Type Abundances - Xenium Mouse Brain', fontsize=16)
plt.tight_layout()
plt.savefig('cell_type_abundances_spatial.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved!")